# 03 - Baseline Models

Train Linear Regression, Random Forest, and XGBoost on the feature matrix.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import polars as pl
from sklearn.preprocessing import StandardScaler

from src.models.baseline import build_linear_regression, build_random_forest, build_xgboost
from src.models.train import train_sklearn_model, evaluate_model
from src.evaluation.metrics import comparison_table

In [ ]:
df_feat = pl.read_parquet('../data/processed/hourly_demand.parquet')

In [ ]:
feature_cols = [c for c in df_feat.columns if c not in ('pickup_hour', 'PULocationID', 'demand', 'pickup_zone', 'borough')]
df_feat = df_feat.drop_nulls().sort('pickup_hour')
n = df_feat.height
X = df_feat.select(feature_cols).to_numpy()
y = df_feat.select('demand').to_numpy().ravel()

train_end, val_end = int(n * 0.7), int(n * 0.85)
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [ ]:
lr = train_sklearn_model(build_linear_regression(), X_train_s, y_train)
rf = train_sklearn_model(build_random_forest(), X_train_s, y_train)
xgb = train_sklearn_model(build_xgboost(), X_train_s, y_train)

In [ ]:
results = {
    'LinearRegression': evaluate_model(lr, X_test_s, y_test),
    'RandomForest': evaluate_model(rf, X_test_s, y_test),
    'XGBoost': evaluate_model(xgb, X_test_s, y_test),
}
comparison_table(results)